> 在线讲义：[第 3 章 · 优化器：SGD → Momentum → Adam → AdamW，每一步怎么挪](https://wikieden.github.io/beyond-transformer/lessons/03-optimizers.html) · 本书《超越 Transformer》

# 第 3 章 · 优化器：SGD → Momentum → Adam → AdamW，每一步怎么挪

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/03-optimizers.html) · 仓库里 `beyond-transformer/notes/lessons/03-optimizers.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 SGD | θ − αg | `run("sgd", lr=...)` | 0.05 / 0.15 收敛，0.21 发散 |
| 第二步 Momentum | v ← βv + g | `run("momentum", lr=0.02)` | f = 0.0074 vs SGD 0.0663 |
| 第三步 偏差修正 | m/(1 − β₁ᵗ) | `adam_first_step(4.0)` | m̂ = 4、v̂ = 16 |
| 第三步 Adam | m̂/√v̂ | `run("adam", lr=0.1)` | 前 3 步沿对角线 |
| 第四步 衰减 | AdamW vs L2 | `decay_compare(wd=...)` | 0.9048 vs 0.2244 |
| 第五步 状态 | 16 B / 参数 | `state_bytes(1e8)` | 1.6 GB |

Adam 的偏差修正；AdamW 和 "Adam + L2" 的区别；每个参数要存几份状态。
这里只用标量和二维向量演示公式；a1 的 AdamW（torch.optim.Optimizer 子类）留给自己写（T-M0 之后的 §4.3）。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
A = np.array([10.0, 1.0])          # f(x, y) = ½(10x² + y²)：x 方向陡、y 方向缓
START = np.array([1.0, 1.0])

**函数 `f`**：小工具。

In [ ]:
def f(p):
    return float(0.5 * (A * p * p).sum())

**函数 `grad`**：小工具。

In [ ]:
def grad(p):
    return A * p

**函数 `run`**：返回走过的点。method：sgd / momentum / adam。

In [ ]:
def run(method: str, steps: int = 50, lr: float = 0.15, beta: float = 0.9, b1: float = 0.9, b2: float = 0.999, eps: float = 1e-8):
    """返回走过的点。method：sgd / momentum / adam。"""
    p, v, m, s = START.copy(), np.zeros(2), np.zeros(2), np.zeros(2)
    path = [p.copy()]
    for t in range(1, steps + 1):
        g = grad(p)
        if method == "sgd":
            p = p - lr * g
        elif method == "momentum":
            v = beta * v + g
            p = p - lr * v
        else:
            m = b1 * m + (1 - b1) * g
            s = b2 * s + (1 - b2) * g * g
            mh, sh = m / (1 - b1**t), s / (1 - b2**t)
            p = p - lr * mh / (np.sqrt(sh) + eps)
        path.append(p.copy())
    return np.array(path)

**函数 `adam_first_step`**：第 1 步：不修正时 m = 0.1g、v = 0.001g²；修正后 m̂ = g、v̂ = g²，步长 = lr · sign(g)。

In [ ]:
def adam_first_step(g: float, b1: float = 0.9, b2: float = 0.999):
    """第 1 步：不修正时 m = 0.1g、v = 0.001g²；修正后 m̂ = g、v̂ = g²，步长 = lr · sign(g)。"""
    m, v = (1 - b1) * g, (1 - b2) * g * g
    return m, v, m / (1 - b1), v / (1 - b2)

**函数 `decay_compare`**：梯度恒为 0（只看衰减）。AdamW：θ ← θ − lr·λ·θ；Adam + L2：把 λθ 加进梯度，再被 √v̂ 除掉。

In [ ]:
def decay_compare(theta: float = 1.0, g: float = 0.0, lr: float = 0.01, wd: float = 0.1, steps: int = 100):
    """梯度恒为 0（只看衰减）。AdamW：θ ← θ − lr·λ·θ；Adam + L2：把 λθ 加进梯度，再被 √v̂ 除掉。"""
    tw, tl = theta, theta
    m = v = 0.0
    for t in range(1, steps + 1):
        tw = tw - lr * wd * tw                  # AdamW：衰减和梯度分开
        gl = g + wd * tl                        # Adam + L2：衰减混进梯度
        m = 0.9 * m + 0.1 * gl
        v = 0.999 * v + 0.001 * gl * gl
        tl = tl - lr * (m / (1 - 0.9**t)) / (np.sqrt(v / (1 - 0.999**t)) + 1e-8)
    return tw, tl

**函数 `state_bytes`**：fp32 训练：参数 4 B + 梯度 4 B + Adam 的 m、v 各 4 B。

In [ ]:
def state_bytes(n_params: int):
    """fp32 训练：参数 4 B + 梯度 4 B + Adam 的 m、v 各 4 B。"""
    return {"参数": 4 * n_params, "梯度": 4 * n_params, "m": 4 * n_params, "v": 4 * n_params}

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("第一步 SGD 在窄长碗里")
for lr in (0.05, 0.15, 0.21):
    pth = run("sgd", lr=lr)
    print(f"  lr = {lr}：50 步后 f = {f(pth[-1]):.3g}，x 方向的系数 1 − 10·lr = {1 - 10 * lr:+.2f}")
print("\n第二步 Momentum")
for name in ("sgd", "momentum"):
    pth = run(name, lr=0.02)
    print(f"  {name}（lr = 0.02）：50 步后 f = {f(pth[-1]):.5f}，点 {np.round(pth[-1], 4).tolist()}")

In [ ]:
print("\n第三步 Adam")
m, v, mh, vh = adam_first_step(4.0)
print(f"  g = 4：不修正 m = {m:.2f}、v = {v:.4f}；修正后 m̂ = {mh:.2f}、v̂ = {vh:.2f}，第一步走 lr·m̂/√v̂ = lr × {mh / np.sqrt(vh):.2f}")
pth = run("adam", lr=0.1)
print(f"  Adam（lr = 0.1）：前 3 步 {np.round(pth[1:4], 3).tolist()}，50 步后 f = {f(pth[-1]):.5f}")

In [ ]:
print("\n第四步 AdamW vs Adam + L2（梯度为 0，只看衰减，100 步）")
for wd in (0.1, 0.01):
    tw, tl = decay_compare(wd=wd)
    print(f"  λ = {wd}：AdamW θ = {tw:.4f}；Adam + L2 θ = {tl:.4f}")

In [ ]:
print("\n第五步 每个参数存几份")
b = state_bytes(100_000_000)
print(f"  1 亿参数：" + "，".join(f"{k} {v / 1e9:.1f} GB" for k, v in b.items()) + f"，合计 {sum(b.values()) / 1e9:.1f} GB")

## 3. 代码里才看得到的

- **t 从 1 数**：t = 0 时 1 − β₁⁰ = 0，会除零。a1 的 Algorithm 1 也特意写了“t starts at 1”。
- **这里只是标量 / 二维的演示**：作业要实现的是 `torch.optim.Optimizer` 的子类，状态按参数存在 `self.state[p]` 里，那部分留给自己写。

## 4. 自己改一改

- 把 `run` 的起点改成 (1, 0)（只有陡的方向），SGD 和 Adam 谁更快？
- 把 Adam 的 β₂ 改成 0.95，50 步后的 f 怎么变？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 窄长山谷 SGD，陡的方向卡步长，缓的方向慢慢挪。
> 动量像铁球，同向越滚越快，来回互相抵消。
> Adam 每参数一把尺，梯度平方量陡缓；开头被零拖小，修正除以一减 β 的 t 次方。
> 衰减别混进梯度，AdamW 单独减 αλθ。
> 参数梯度加两矩，十六字节一个参数。